# RigTech · runner_colab

Este notebook **não contém lógica**. Ele apenas clona o repo, prepara o ambiente,
traz os dados do Drive para o disco local e chama os scripts em `src/`.

Se você se pegar escrevendo lógica dentro de uma célula, leve para `src/` e faça commit.

## 1. Ambiente

In [ ]:
!git clone https://github.com/SEU_ORG/rigtech-weed-cycle.git
%cd rigtech-weed-cycle
!pip install -q -r requirements.txt

## 2. Montar o Drive e trazer o dataset para o disco local

**Nunca treine lendo o Drive montado** — a latência por arquivo mata o dataloader.
Sempre: tarball -> copiar -> extrair em `/content/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# TODO(fase-1): apontar para a pasta real no Drive.
!cp -r /content/drive/MyDrive/rigtech/live/. work/live/
!cp -r /content/drive/MyDrive/rigtech/golden/. work/golden/
!cp -r /content/drive/MyDrive/rigtech/versions/. work/versions/ 2>/dev/null || true

## 3. QA estático (CPU, segundos)

In [ ]:
!python -m src.qa_static

## 4. Rodada de detecção de suspeitas + criação dos cards

In [ ]:
import os
os.environ['LINEAR_API_KEY'] = 'lin_api_...'   # TODO(fase-1): usar Colab Secrets

!python -m src.run_cycle flag --cycle 1 --dry-run   # inspecionar
# !python -m src.run_cycle flag --cycle 1           # criar os cards

## 5. Watcher — deixar de pé enquanto durar a sessão

In [ ]:
!python -m src.run_cycle watch --interval 300

## 6. Treino avulso (baseline ou comparação de arquiteturas)

In [ ]:
!python -m src.snapshot --note 'baseline inicial'
!python -m src.run_cycle train --version v1 --tag baseline

## 7. Histórico

In [ ]:
!cat work/runs/history.json

## 8. Devolver versões e runs ao Drive ANTES de a sessão morrer

(a etapa mais frequentemente esquecida — sem isso, tudo se perde ao encerrar o Colab)

In [ ]:
!rsync -av work/versions/ /content/drive/MyDrive/rigtech/versions/
!rsync -av work/runs/     /content/drive/MyDrive/rigtech/runs/